# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring.** I'm picking this lane because I've already worked hands-on with exactly this problem in the Week 1–2 starter notebooks  building a hand-rule score, comparing it against a learned model, and hitting the leakage trap firsthand when trend_pct was fed in as a feature. That gave me a working sense of what "good" looks like here: a ranked queue with reason codes a reviewer can actually act on, not just a raw prediction. The lane also has the deepest, most reliable data (fact_content_daily_performance with 78.8M rows, 28.9M with GSC impressions), so I won't be fighting sparse signal the way the AI-referral freestyle direction would

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [7]:
!git clone https://github.com/EimanZahra1472/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 38), reused 98 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 17.07 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/flyrank-ml-internship-starter


In [8]:
import os
print(os.getcwd())
print(os.listdir())

/content/flyrank-ml-internship-starter
['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs', '.git', 'AGENTS.md', 'LICENSE', '.github', 'submission', 'CLAUDE.md', 'GUIDE.md', '.gitignore', 'requirements.txt']


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages in a client's content inventory should be reviewed first for refresh, given limited reviewer time.
Who acts: A content strategist or SEO reviewer who can manually check maybe 20–50 pages per cycle, not the full inventory.

**Action taken:** The reviewer opens the top-ranked pages, reads the reason code (e.g. "stale and visible," "declining with demand"), and decides whether to rewrite, expand, consolidate, or leave the page alone.

**Cost of a wrong call:** A false positive (flagging a healthy page) wastes limited reviewer time that could've gone to a genuinely declining page. A false negative (missing a real decline) means a page keeps losing visibility unnoticed until it's harder to recover. Given limited capacity, precision at the top of the ranked list matters more than catching every possible case  which is why Precision@K, not overall accuracy, is the right metric here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))
print(os.listdir("data/raw"))

True
['content_refresh_anonymized.csv']


In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(df.shape)

(30000, 44)


In [16]:
# Number 1: how much of the inventory is currently declining?
decline_rate = (df["trend_direction"] == "down").mean()
print(f"Declining rate: {decline_rate:.3f}  ({(df['trend_direction']=='down').sum()} of {len(df)} pages)")

# Number 2: how many pages meet the 'stale and visible' reason code on their own?
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale + visible pages: {stale_visible} ({stale_visible/len(df):.3f} of inventory)")

# Number 3: the model-vs-baseline lift already proven in the starter pipeline
print("Baseline rule Precision@50: 0.240")
print("Random forest Precision@50: 0.740")
print("~3x improvement in how many of the top 50 flagged pages are truly declining.")

Declining rate: 0.542  (16262 of 30000 pages)
Stale + visible pages: 17 (0.001 of inventory)
Baseline rule Precision@50: 0.240
Random forest Precision@50: 0.740
~3x improvement in how many of the top 50 flagged pages are truly declining.


On the 30,000-page starter slice, 54.2% of pages (16,262) are currently declining (trend_direction == "down")  a high base rate that makes this a real, common problem across the inventory, not a rare edge case.

Interestingly, only 17 pages (0.1% of the inventory) meet the simplest possible reason code "stale and visible" (days_since_last_update >= 180 and impressions_90d >= 500)  on their own. That's a strong signal that a single hand-written rule is too narrow to build a useful review queue: most of the declining inventory won't get flagged by that rule alone, which is exactly why a learned, multi-signal ranking is worth building rather than just hard-coding one threshold.

This is backed up directly by the starter pipeline's own results: the baseline rule scores Precision@50 = 0.240, while a random forest trained on the same observable signals reaches Precision@50 = 0.740 roughly a 3x improvement in how many of the top 50 flagged pages are genuinely declining. That gap is the concrete evidence that this lane rewards a model over a fixed rule, and is worth the next 7 weeks of work.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work can say: which pages, based on observed 90-day signals (impressions, position, staleness, CTR), show a statistical association with decline or opportunity; a ranked, evidence-backed review queue with reason codes a human can inspect; a measured comparison of a learned ranking against a transparent baseline rule, validated on held-out clients.

What this work cannot say: that refreshing a flagged page will cause it to recover  that would require a real experiment, not observational data; that any signal reflects Google's actual ranking algorithm; that a page is "definitely" declining rather than showing an association with decline in this dataset. Every claim in this project will use observed, measured, or directional language  this is decision-support for a human reviewer, not an automated verdict.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.